# Decision Tree — Strategy A: SMOTE

This notebook trains and evaluates a Decision Tree classifier using Strategy A for class imbalance handling. In this strategy, SMOTE is applied to the preprocessed training set to balance the minority diabetes class.

The model is trained on the SMOTE-balanced training data and evaluated on the original untouched test set. The results are saved for later comparison with the class-weighted Decision Tree model and the other machine learning models.

In [1]:
import pandas as pd

In [2]:
from pathlib import Path

X_train_final = pd.read_parquet("../DATASETS/PREPROCESSED/X_train_final.parquet")
X_test_final = pd.read_parquet("../DATASETS/PREPROCESSED/X_test_final.parquet")

y_train = pd.read_parquet("../DATASETS/PREPROCESSED/y_train.parquet")["diabetes"]
y_test = pd.read_parquet("../DATASETS/PREPROCESSED/y_test.parquet")["diabetes"]

In [3]:
# SMOTE Balancing

from imblearn.over_sampling import SMOTE

X_train_final_smote, y_train_smote = SMOTE(random_state=42).fit_resample(X_train_final, y_train)


X_train_final_smote

,age,income_poverty_ratio,bmi,waist_circumference,mean_systolic_bp,mean_diastolic_bp,average_alcoholic_drinks_per_day,sleep_hours,energy_kcal,protein_g,...,moderate_work_activity_7.0,moderate_work_activity_9.0,walk_or_bicycle_transport_2.0,walk_or_bicycle_transport_7.0,walk_or_bicycle_transport_9.0,vigorous_recreational_activity_2.0,vigorous_recreational_activity_7.0,moderate_recreational_activity_2.0,moderate_recreational_activity_7.0,moderate_recreational_activity_9.0
0,-0.515966,-0.820113,-0.394755,-0.382896,0.588800,0.645527,-0.041070,0.442632,-0.027411,-0.369739,...,False,False,True,False,False,True,False,True,False,False
1,-0.346237,-0.826584,1.086548,0.313449,0.304926,1.041975,-0.041070,-0.132430,-0.137124,-0.422362,...,False,False,True,False,False,True,False,True,False,False
2,-1.251459,-0.101758,-0.095646,0.117206,-0.963045,0.249079,-0.078853,0.634319,-0.486502,-0.030426,...,False,False,True,False,False,True,False,False,False,False
3,1.351055,1.645589,-0.551431,0.281797,-0.281747,-1.959701,-0.041070,0.634319,0.975986,1.212153,...,False,False,True,False,False,False,False,True,False,False
4,0.162951,0.506577,-1.477245,-1.313465,0.626650,0.475621,-0.078853,0.250945,0.849230,0.806813,...,False,False,True,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33741,1.443335,-0.750876,-0.005785,0.327342,2.018200,0.846613,-0.075811,1.354761,-1.432120,-0.386123,...,False,False,True,False,False,True,False,True,False,False
33742,0.589388,1.470840,-0.593946,-0.180514,0.831153,0.509431,-0.073777,-0.029425,0.427500,0.314823,...,False,False,True,False,False,True,False,False,False,False
33743,1.493449,0.245480,1.276673,1.169856,1.551492,1.135095,-0.041070,0.795135,-0.891120,-0.397454,...,False,False,True,False,False,True,False,True,False,False
33744,0.401165,-1.355167,0.681098,0.448197,0.008528,0.301030,-0.061524,0.043401,-1.424249,-1.038774,...,False,False,True,False,False,True,False,True,False,False


In [4]:
# Training

from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    random_state=42,
)

model.fit(X_train_final_smote, y_train_smote)

y_pred = model.predict(X_test_final)
y_pred_proba = model.predict_proba(X_test_final)[:, 1]

In [5]:
# Plotting confusion matrix

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

conf_matrix = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix)
disp.plot(ax=ax, colorbar=False, cmap="Blues")

ax.set_title("Confusion Matrix", color="black", fontsize=13, pad=12)
ax.set_xlabel("Predicted", color="black")
ax.set_ylabel("Actual", color="black")
ax.tick_params(colors="black")

for text in disp.text_.ravel():
    text.set_color("black")
    text.set_fontsize(14)
disp.text_[0, 0].set_color("white")

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
%pip install matplotlib

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for package-name: filename=package_name-0.1-py3-none-any.whl size=1249 sha256=35bf94ca0d648c6db27427028063fde189cbb38d2f49eee5b3eece85438bb821
  Stored in directory: c:\users\jayde\appdata\local\pip\cache\wheels\b3\c1\6f\538e951eb00f535f43151173b4c55e463a35c17b9e90ab3b1a
Successfully built package-name
Note: you may need to restart the kernel to use updated packages.


In [13]:
# results

from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")
print("Confusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7349
ROC AUC: 0.6261
Confusion Matrix:
[[3379  840]
 [ 539  443]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.86      0.80      0.83      4219
         1.0       0.35      0.45      0.39       982

    accuracy                           0.73      5201
   macro avg       0.60      0.63      0.61      5201
weighted avg       0.76      0.73      0.75      5201

